# Location Mapping Viewer
Map raw location text -> ontology world location แล้วดูคุณภาพผลลัพธ์


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)


In [ ]:
# รัน mapping pipeline (รองรับทั้งกรณีเปิด notebook จาก project root หรือจาก data/processed)
from pathlib import Path
import subprocess

candidates = [Path('location_mapping_pipeline.py'), Path('../../location_mapping_pipeline.py')]
script = next((p for p in candidates if p.exists()), None)
if script is None:
    raise FileNotFoundError('หา location_mapping_pipeline.py ไม่เจอจากตำแหน่ง notebook ปัจจุบัน')

subprocess.run([
    'python3', str(script),
    '--profiles-file', 'data/processed/all_profiles_cleaned.csv' if Path('data/processed/all_profiles_cleaned.csv').exists() else '../processed/all_profiles_cleaned.csv',
    '--ontology-file', 'data/ontology/location_ontology.csv' if Path('data/ontology/location_ontology.csv').exists() else '../ontology/location_ontology.csv',
    '--output-file', 'data/processed/location_mapping.csv' if Path('data/processed').exists() else 'location_mapping.csv',
    '--threshold', '0.75'
], check=True)


In [ ]:
from pathlib import Path
mapping_candidates = [Path('data/processed/location_mapping.csv'), Path('location_mapping.csv')]
mapping_path = next((p for p in mapping_candidates if p.exists()), None)
if mapping_path is None:
    raise FileNotFoundError('หา location_mapping.csv ไม่เจอ')

df = pd.read_csv(mapping_path)
print('using', mapping_path)
print(df.shape)
df.head(10)


In [ ]:
summary = {
    'total_rows': len(df),
    'matched_rows': int((df['match_method'] != 'no_match').sum()),
    'no_match_rows': int((df['match_method'] == 'no_match').sum()),
    'mean_confidence': round(float(df['confidence'].mean()), 4),
}
summary


In [ ]:
df['match_method'].value_counts()


In [ ]:
# Top confidence
df.sort_values('confidence', ascending=False).head(20)


In [ ]:
# Lowest confidence ที่ยัง match
df[df['match_method'] != 'no_match'].sort_values('confidence', ascending=True).head(20)


In [ ]:
# ไม่ match เพื่อเอาไปปรับ ontology ต่อ
df[df['match_method'] == 'no_match'].head(30)


In [ ]:
plt.figure(figsize=(8,4))
df['confidence'].hist(bins=30)
plt.title('Confidence Distribution')
plt.xlabel('confidence')
plt.ylabel('count')
plt.show()
